In [ ]:
import csv
import dataclasses
import hashlib
import json
import os
import sys
import textwrap
from datetime import datetime
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from scipy.ndimage import gaussian_filter1d
from IPython.display import Image, display

print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | Matplotlib {matplotlib.__version__}')

In [ ]:
# Notebook may be opened from either the repository root or figures/.
candidates = [Path.cwd(), *Path.cwd().parents]
REPO_ROOT = next((p for p in candidates if (p / 'figures/figdata/med_shap/config.json').is_file()),
                 Path('/home/hsh/GPT-Disease-Drug-Prediction'))
FIGURES_DIR = REPO_ROOT / 'figures'
SOURCE_DIR = FIGURES_DIR / 'figdata/med_shap'
QC_DIR = SOURCE_DIR / 'notebook_qc'
FORMATS = ('png', 'pdf', 'svg')

# Notebook assets are kept locally under figures/figutil.
FONT_PATH = FIGURES_DIR / 'figutil/Helvetica.ttf'
BOLD_FONT_PATH = FIGURES_DIR / 'figutil/Helvetica Bold.ttf'
for _path in (FONT_PATH, BOLD_FONT_PATH):
    assert _path.is_file(), _path
    fm.fontManager.addfont(_path)
FONT_NAME = fm.FontProperties(fname=FONT_PATH).get_name()
# 'Helvetica Bold.ttf' registers as family 'Helvetica75'; re-tag it as bold
# 'Helvetica' so weight='bold' text resolves to the bold file.
for _i, _entry in enumerate(fm.fontManager.ttflist):
    if _entry.fname == str(BOLD_FONT_PATH):
        fm.fontManager.ttflist[_i] = dataclasses.replace(_entry, name=FONT_NAME, weight=700)
plt.rcParams['font.family'] = FONT_NAME
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = FONT_NAME
plt.rcParams['mathtext.bf'] = f'{FONT_NAME}:bold'
assert fm.findfont(fm.FontProperties(weight='bold')) == str(BOLD_FONT_PATH)

# Plotting parameters. The SHAP permutations and vertical packing use seed 9995.
LOW, MID, HIGH = '#2E76AD', '#E4E2E3', '#C34D60'
ABSENT = '#C8CDD2'
CMAP = LinearSegmentedColormap.from_list('pastel_blue_red_enhanced',
        [(0.,LOW),(.22,'#78A9D1'),(.5,MID),(.78,'#DE8B91'),(1.,HIGH)])
HALF_WIDTH = .43
DOT_SIZE = 6.0
DOT_ALPHA = .78
DENSITY_FILL = '#AEB5BF'
DENSITY_EDGE = '#7D8794'
DENSITY_ALPHA = .60
DENSITY_EDGE_ALPHA = .88
DENSITY_LINEWIDTH = .90
SWARM_ENVELOPE_FRACTION = .82
AXIS_MARGIN = 1.08
GATE_FIGSIZE = (9.5,13)
GRID_FIGSIZE = (16,30)
GATE_LAYOUT = dict(left=.44,right=.975,bottom=.15,top=.82)
GRID_LAYOUT = dict(left=.24,right=.985,bottom=.085,top=.88,hspace=.70,wspace=.90)

assert SOURCE_DIR.is_dir(), SOURCE_DIR
QC_DIR.mkdir(parents=True, exist_ok=True)
print('Source:', SOURCE_DIR)
print('Figure QC data:', QC_DIR)
print('Figure file export: disabled until final approval')
print('Font:', FONT_NAME, '|', FONT_PATH)

In [ ]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + '\n')
    os.replace(tmp, path)

In [ ]:
config = json.loads((SOURCE_DIR / 'config.json').read_text())
source_qc = json.loads((SOURCE_DIR / 'qa.json').read_text())
assert source_qc['status'] == 'COMPLETE'
importance_table = pd.read_csv(SOURCE_DIR / 'feature_importance.csv')

def attribution_feature_id(label):
    ids = importance_table.loc[importance_table['feature'] == label, 'feature_id'].unique()
    assert len(ids) == 1, (label, ids)
    return int(ids[0])

AGE = attribution_feature_id('Age at prediction (continuous)')
SEX = attribution_feature_id('Female (reference: male)')
NOUTPUT = len(config['outputs'])
shap_values = np.load(SOURCE_DIR / 'shap.npy', mmap_mode='r')
feature_values = np.load(SOURCE_DIR / 'feature_values.npy', mmap_mode='r')
with (SOURCE_DIR / 'labels.csv').open() as handle:
    feature_names = [','.join(row).rstrip(',') for row in csv.reader(handle)]
feature_names += ['Unused', 'Female (reference: male)', 'Age at prediction (continuous)']

assert shap_values.shape == (config['explained_patients'], len(feature_names), NOUTPUT)
assert feature_values.shape == shap_values.shape[:2]
assert not shap_values.flags.writeable and not feature_values.flags.writeable
assert feature_names[AGE] == 'Age at prediction (continuous)'
assert feature_names[SEX] == 'Female (reference: male)'
assert feature_names[2].lower() == 'female' and feature_names[3].lower() == 'male'
sex_values = feature_values[:, SEX]
assert set(np.unique(sex_values)) == {0., 1.}
assert np.all(shap_values[sex_values == 0, SEX, :] == 0)
source_files = ['shap.npy', 'feature_values.npy', 'feature_importance.csv']
source_hashes = {name: sha256(SOURCE_DIR / name) for name in source_files}

display(pd.DataFrame([{
    'patients / transitions': len(shap_values),
    'outputs': NOUTPUT,
    'male reference': int((sex_values == 0).sum()),
    'female': int((sex_values == 1).sum()),
    'checkpoint step': config['checkpoint_step'],
}]))
print('Checkpoint:', config['checkpoint'])
print('Data:', config['data'])

In [ ]:
def compact(name):
    name = name.replace('non-insulin-dependent diabetes mellitus', 'type 2 diabetes mellitus')
    name = name.replace('insulin-dependent diabetes mellitus', 'type 1 diabetes mellitus')
    return name if len(name) <= 48 else name[:45].rstrip() + '…'


def check_text_bounds(fig):
    from matplotlib.text import Text
    fig.canvas.draw(); renderer = fig.canvas.get_renderer(); omitted = set(); bad = []
    all_axes = list(fig.axes) + [child for ax in fig.axes for child in ax.child_axes]
    for ax in all_axes:
        for axis, limits in [(ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())]:
            lo, hi = sorted(limits)
            for tick in axis.get_major_ticks() + axis.get_minor_ticks():
                if not lo <= tick.get_loc() <= hi: omitted.update([id(tick.label1), id(tick.label2)])
    for item in fig.findobj(Text):
        if id(item) in omitted or not item.get_visible() or not item.get_text(): continue
        if item.axes is not None and item.get_clip_on(): continue
        box = item.get_window_extent(renderer)
        if box.width and box.height and (box.x0 < -2 or box.y0 < -2 or box.x1 > fig.bbox.width + 2 or box.y1 > fig.bbox.height + 2):
            bad.append(item.get_text())
    assert not bad, bad

In [ ]:
def feature_colors(raw,g):
    """Display normalization only; returns colors and auditable limits in raw units."""
    raw=np.asarray(raw,dtype=float)
    assert np.isfinite(raw).all() and (raw>=0).all()
    info=dict(feature_id=int(g),n=len(raw),zero_n=int((raw==0).sum()))
    if g==SEX:
        assert set(np.unique(raw)) <= {0.,1.}
        return np.array([matplotlib.colors.to_rgba(LOW if x==0 else HIGH) for x in raw]),dict(info,mode='sex_binary',male_n=int((raw==0).sum()),female_n=int((raw==1).sum()))
    if g==AGE:
        low,high=float(raw.min()),float(raw.max())
        z=(raw-low)/(high-low) if high>low else np.full(len(raw),.5)
        return CMAP(z),dict(info,mode='continuous_age_minmax',low=low,high=high)
    present=raw>0
    colors=np.tile(matplotlib.colors.to_rgba(ABSENT),(len(raw),1))
    if not present.any(): return colors,dict(info,mode='history_all_absent')
    x=raw[present]
    low,high=np.quantile(x,[.05,.95])
    mode='positive_history_log1p_p05_p95'
    if high<=low:
        low,high=x.min(),x.max()
        mode='positive_history_log1p_minmax_collapsed_quantiles'
    if high>low:
        z=np.clip((np.log1p(x)-np.log1p(low))/(np.log1p(high)-np.log1p(low)),0,1)
    else:
        z=np.full(len(x),.5)
        mode='constant_positive_history_midcolor'
    colors[present]=CMAP(z)
    return colors,dict(info,mode=mode,positive_n=int(present.sum()),low=float(low),high=float(high),
                       below_low_n=int((x<low).sum()),above_high_n=int((x>high).sum()))

In [ ]:
def expanded_offsets(x,rng,width=HALF_WIDTH):
    """Same x values; square-root bin occupancy controls only vertical packing."""
    if np.ptp(x)<1e-12: return rng.uniform(-width*.32,width*.32,len(x))
    edges=np.linspace(x.min()-1e-12,x.max()+1e-12,81)
    bins=np.clip(np.digitize(x,edges)-1,0,79)
    counts=np.bincount(bins,minlength=80)
    yy=np.zeros(len(x))
    for b in np.flatnonzero(counts):
        ix=np.flatnonzero(bins==b);rng.shuffle(ix)
        if len(ix)>1:
            yy[ix]=np.linspace(-width,width,len(ix))*np.sqrt(len(ix)/counts.max())
    return yy


def density(ax,x,y,color,side=0):
    """One full-range envelope also used to constrain point-center packing."""
    if len(x)<3 or np.ptp(x)<1e-9:
        if len(x): ax.plot([np.median(x)]*2,[y-.10,y+.10],color=color,lw=1.0)
        return None
    padding=float(np.ptp(x))*.015
    counts,edges=np.histogram(x,bins=100,range=(x.min()-padding,x.max()+padding))
    assert counts.sum()==len(x),'Histogram dropped an observation'
    smooth=gaussian_filter1d(counts.astype(float),1.5)
    h=HALF_WIDTH*np.sqrt(smooth/max(smooth.max(),1))
    xx=(edges[1:]+edges[:-1])/2
    # Close outside the observed extrema so the extreme observations are included.
    xx=np.r_[edges[0],xx,edges[-1]]
    h=np.r_[0.,h,0.]
    lo,hi=(y-h,y+h) if side==0 else ((y-h,np.full_like(h,y)) if side<0 else (np.full_like(h,y),y+h))
    gray=color==DENSITY_FILL
    ax.fill_between(xx,lo,hi,color=color,alpha=DENSITY_ALPHA if gray else .36,lw=0,zorder=1)
    edge=DENSITY_EDGE if gray else color
    ax.plot(xx,lo,color=edge,alpha=DENSITY_EDGE_ALPHA,lw=DENSITY_LINEWIDTH,zorder=2)
    ax.plot(xx,hi,color=edge,alpha=DENSITY_EDGE_ALPHA,lw=DENSITY_LINEWIDTH,zorder=2)
    return xx,h


def pack_inside_envelope(x,rng,envelope):
    offsets=expanded_offsets(x,rng)
    if envelope is not None:
        half_width=np.interp(x,*envelope)
        bound=half_width*SWARM_ENVELOPE_FRACTION
        offsets=np.clip(offsets,-bound,bound)
        assert np.all(np.abs(offsets)<=half_width+1e-12)
    return offsets

In [ ]:
def panel(ax,sv,values,names,title,letter,k,top=20,title_y=1.17):
    importance=np.abs(sv[:,:,k]).mean(0)
    order=np.argsort(-importance,kind='stable')[:top]
    order=order[importance[order]>0]
    limit=max(.7,float(np.abs(sv[:,order,k]).max())*AXIS_MARGIN)
    ax.set_xlim(-limit,limit)
    rng=np.random.default_rng(9995)
    display_rows=[(g,state) for g in order for state in ([0,1] if g==SEX else [None])]
    labels=[]
    row_qc=[]
    clouds=[]
    for y,(g,state) in enumerate(display_rows):
        x,raw=np.asarray(sv[:,g,k]),np.asarray(values[:,g])
        assert np.isfinite(x).all()
        if g==SEX:
            v=x[raw==state];color=LOW if state==0 else HIGH
            labels.append(f'Sex: male reference, SHAP=0 (n={len(v):,})' if state==0 else f'Sex: female vs male (n={len(v):,})')
            if not len(v):continue
            if state==0: assert np.all(v==0),'Male reference SHAP unexpectedly nonzero'
            envelope=density(ax,v,y,color)
            dy=pack_inside_envelope(v,rng,envelope)
            if state==0:
                # This is a point mass at zero, not a continuous male distribution.
                dy=rng.uniform(-HALF_WIDTH*.82,HALF_WIDTH*.82,len(v))
            cloud=ax.scatter(v,y+dy,s=DOT_SIZE,c=color,alpha=DOT_ALPHA,lw=0,zorder=4)
            expected_x=v
            if state==0:
                ax.scatter([0],[y],s=24,marker='D',facecolors='white',edgecolors=LOW,lw=1,zorder=5)
        else:
            labels.append(compact(names[g]))
            envelope=density(ax,x,y,DENSITY_FILL)
            colors,_=feature_colors(raw,g)
            dy=pack_inside_envelope(x,rng,envelope)
            ix=rng.permutation(len(x))
            if g!=AGE:
                # Keep all absent histories, drawn behind the nonzero histories.
                ix=np.r_[ix[raw[ix]==0],ix[raw[ix]>0]]
            cloud=ax.scatter(x[ix],y+dy[ix],s=DOT_SIZE,c=colors[ix],alpha=DOT_ALPHA,lw=0,zorder=3)
            expected_x=x
        offsets=np.asarray(cloud.get_offsets())
        assert len(offsets)==len(expected_x)
        assert np.array_equal(np.sort(offsets[:,0]),np.sort(expected_x)),'Horizontal SHAP values changed'
        outside_envelope=0
        if envelope is not None:
            widths=np.interp(offsets[:,0],*envelope)
            outside_envelope=int((np.abs(offsets[:,1]-y)>widths+1e-10).sum())
            assert outside_envelope==0,'Point center lies outside density envelope'
        row_qc.append(dict(feature_id=int(g),state=state,expected_points=len(expected_x),drawn_points=len(offsets),
                           xmin=float(expected_x.min()),xmax=float(expected_x.max()),
                           outside_envelope=outside_envelope,envelope_applies=envelope is not None))
        clouds.append(cloud)
    ax.axvline(0,color='#69767A',ls=(0,(3,3)),lw=.7)
    ax.set_yticks(range(len(display_rows)),labels,fontsize=8)
    ax.set_ylim(len(display_rows)-.4,-.7)
    ax.set_xlabel('SHAP contribution (log-odds)',fontsize=10,labelpad=7)
    ax.tick_params(labelsize=8)
    ax.spines[['top','right']].set_visible(False)
    sec=ax.secondary_xaxis('top',functions=(np.exp,lambda x:np.log(np.maximum(x,1e-300))))
    # Equal distances on this auxiliary scale correspond to logarithms of odds multipliers.
    step=max(1,int(np.ceil(limit/np.log(10)/3)))
    powers=np.arange(-30,31,step)
    powers=powers[abs(powers*np.log(10))<limit]
    ticks=10.**powers
    assert np.allclose(np.log(ticks),powers*np.log(10))
    sec.set_xticks(ticks,[f'{t:g}' for t in ticks])
    sec.set_xlabel('Odds multiplier: exp(SHAP)',fontsize=8,labelpad=7)
    sec.tick_params(labelsize=8)
    ax.text(-.025,title_y,letter,transform=ax.transAxes,fontsize=15,weight='bold')
    ax.text(.07,title_y,title,transform=ax.transAxes,fontsize=10,weight='bold')
    assert sum(r['drawn_points'] for r in row_qc)==len(sv)*len(order)
    ax._shap_clouds=clouds
    ax._shap_qc=dict(output_index=k,features=len(order),patients=len(sv),rows=row_qc,
                     expected_points=len(sv)*len(order),drawn_points=sum(r['drawn_points'] for r in row_qc),
                     x_limits=[-limit,limit])
    return order.tolist()


def export(fig,out,stem,formats=('png','pdf','svg')):
    check_text_bounds(fig)
    fig.canvas.draw()
    point_qc=[]
    for ax in fig.axes:
        if not hasattr(ax,'_shap_qc'):continue
        outside_axes=0
        for cloud in ax._shap_clouds:
            coords=ax.transData.transform(np.asarray(cloud.get_offsets()))
            radius=float(np.sqrt(cloud.get_sizes().max())*.5*fig.dpi/72)
            outside_axes+=int(((coords[:,0]-radius<ax.bbox.x0)|(coords[:,0]+radius>ax.bbox.x1)|
                              (coords[:,1]-radius<ax.bbox.y0)|(coords[:,1]+radius>ax.bbox.y1)).sum())
        assert outside_axes==0,'Scatter marker clipped by axes'
        point_qc.append(dict(ax._shap_qc,markers_outside_axes=outside_axes))
    write_json(out/f'{stem}_point_coverage.json',dict(status='PASS',panels=point_qc,
               density='Square-root smoothed histogram frequency, all samples; not a confidence interval',
               point_centers_inside_envelope=True,all_selected_feature_observations_drawn=True))
    # for ext in formats:
    #     fig.savefig(out/f'{stem}.{ext}',dpi=300)
    #     print(f'Saved {stem}.{ext}',flush=True)
    # A lightweight preview is derived from the same plotted data and layout.
    # fig.savefig(out/f'{stem}_preview.png',dpi=100)
    display(fig)
    plt.close(fig)

In [ ]:
def decorate(fig,cfg,n,preview=False):
    cbax=fig.add_axes([.045,.940,.24,.005])
    cb=fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(0,1),cmap=CMAP),cax=cbax,orientation='horizontal')
    cb.set_ticks([0,.5,1],labels=['P5 or lower','Log midpoint','P95 or higher']);cb.ax.tick_params(labelsize=8,length=0)
    fig.text(.045,.951,'History count > 0: log scale, P5-P95',fontsize=9)
    cbax_age=fig.add_axes([.355,.940,.19,.005])
    cb_age=fig.colorbar(plt.cm.ScalarMappable(norm=Normalize(0,1),cmap=CMAP),cax=cbax_age,orientation='horizontal')
    cb_age.set_ticks([0,.5,1],labels=['Min','Midpoint','Max']);cb_age.ax.tick_params(labelsize=8,length=0)
    fig.text(.355,.951,'Continuous age (years)',fontsize=9)
    fig.legend(handles=[Patch(facecolor=ABSENT,label='0 records in analyzed history'),
                        Patch(facecolor=LOW,label='Male / reference (0)'),Patch(facecolor=HIGH,label='Female (1)')],
               loc='upper left',bbox_to_anchor=(.60,.956),ncol=1,frameon=False,fontsize=8)
    prefix='TECHNICAL DRY RUN — not a population finding. ' if preview else ''
    notes=[
        'Colors are feature-specific; all tail values retained. Sex is one feature shown in two rows; male SHAP is zero by construction.',
        prefix+f'n={n:,} patient-transitions | Control step {cfg["checkpoint_step"]:,} | TIME: 16 midpoint quantiles | native uncalibrated outputs.',
        'All patients shown. Points packed within square-root histogram envelopes; x = exact SHAP. Rows rank mean |SHAP|; panel scales differ.',
        'Reference-dependent model associations, not causal treatment effects. Current anchor and pre-context fixed; continuous empirical-age reference.'
    ]
    wrap_width=int((fig.get_figwidth()-.8)*15)
    footer='\n'.join(textwrap.fill(t,width=wrap_width,break_long_words=False,break_on_hyphens=False) for t in notes)
    fig.text(.04,.016,footer,fontsize=8,va='bottom',linespacing=1.4)


def render_figures(sv,vals,names,cfg,dest,preview=False,sets=('gate','all','conditional'),formats=('png','pdf','svg')):
    n=len(sv)
    # pdf.fonttype 3 (as in figures/umap.ipynb): the legacy Helvetica.ttf has a
    # malformed glyf table, so fontTools subsetting under fonttype 42 fails.
    plt.rcParams.update({'font.family':FONT_NAME,'pdf.fonttype':3,'svg.fonttype':'none'})
    if 'gate' in sets:
        fig,ax=plt.subplots(figsize=GATE_FIGSIZE)
        fig.subplots_adjust(**GATE_LAYOUT)
        fig.text(.04,.982,'What drives the medication gate?',fontsize=18,weight='bold',va='top')
        decorate(fig,cfg,n,preview)
        panel(ax,sv,vals,names,'Any supported medication vs nonmedication','A',0,top=25,title_y=1.09)
        export(fig,dest,'medication_gate_shap',formats)
        print('Rendered medication gate',flush=True)
    for conditional in [False,True]:
        if ('conditional' if conditional else 'all') not in sets: continue
        fig,axes=plt.subplots(4,2,figsize=GRID_FIGSIZE)
        fig.subplots_adjust(**GRID_LAYOUT)
        title='Medication gate and seven medication classes' if not conditional else 'Which medication, given a medication event?'
        fig.text(.04,.982,title,fontsize=23,weight='bold',va='top')
        decorate(fig,cfg,n,preview)
        indices=list(range(8)) if not conditional else list(range(8,15))
        for i,(ax,k) in enumerate(zip(axes.flat,indices)):
            name='Medication gate' if k==0 else cfg['medication_labels'][(k-1)%7]
            panel(ax,sv,vals,names,name,chr(65+i),k,top=20)
        if conditional:
            axes.flat[-1].axis('off')
            axes.flat[-1].text(0,.85,'Conditional class probability\n\nP(next event is class m) / P(next event is medication)\n\nAction is summed before TIME integration.\nThese seven probabilities sum to 1.\n\nSHAP values on the log-odds scale do not sum\nacross classes to the gate SHAP.',va='top',fontsize=11,linespacing=1.8)
        export(fig,dest,'medication_conditional_shap' if conditional else 'medication_all_classes_shap',formats)
        print('Rendered conditional classes' if conditional else 'Rendered all classes',flush=True)

In [ ]:
render_figures(shap_values, feature_values, feature_names, config, QC_DIR,
               sets=('gate',), formats=FORMATS)
print('Rendered inline; figure files were not saved.')

## 5. Gate and seven joint medication outputs

Each medication panel explains its joint next-event probability. Top 20 features per panel. Axis ranges vary by panel and retain all tails.

In [ ]:
render_figures(shap_values, feature_values, feature_names, config, QC_DIR,
               sets=('all',), formats=FORMATS)
print('Rendered inline; figure files were not saved.')

## 6. Medication choice, conditional on a medication event

These panels explain relative medication choice after conditioning on a medication event; this is a different target from the Gate and joint-class panels.

In [ ]:
render_figures(shap_values, feature_values, feature_names, config, QC_DIR,
               sets=('conditional',), formats=FORMATS)
print('Rendered inline; figure files were not saved.')

In [ ]:
coverage_rows = []
for path in sorted(QC_DIR.glob('*_point_coverage.json')):
    result = json.loads(path.read_text())
    assert result['status'] == 'PASS'
    for panel_qc in result['panels']:
        assert panel_qc['expected_points'] == panel_qc['drawn_points']
        assert panel_qc['markers_outside_axes'] == 0
        assert all(row['outside_envelope'] == 0 for row in panel_qc['rows'])
    coverage_rows.append({
        'figure': path.stem.replace('_point_coverage', ''),
        'panels': len(result['panels']),
        'points': sum(row['drawn_points'] for row in result['panels']),
        'missing / clipped': 0,
    })
assert len(coverage_rows) == 3
assert source_hashes == {name: sha256(SOURCE_DIR / name) for name in source_files}

selected_ids = sorted({row['feature_id']
                      for path in QC_DIR.glob('*_point_coverage.json')
                      for panel_qc in json.loads(path.read_text())['panels']
                      for row in panel_qc['rows']})
color_scales = []
for feature_id in selected_ids:
    _, scale = feature_colors(feature_values[:, feature_id], feature_id)
    color_scales.append(dict(scale, feature=feature_names[feature_id]))
write_json(QC_DIR / 'feature_color_scales.json', color_scales)

run_qc = dict(status='PASS', patients=len(shap_values), source=str(SOURCE_DIR),
              output=str(QC_DIR), figure_export='disabled', source_sha256=source_hashes,
              source_arrays_unchanged=True, all_points_retained=True,
              gate_figsize=GATE_FIGSIZE, grid_figsize=GRID_FIGSIZE,
              gate_layout=GATE_LAYOUT, grid_layout=GRID_LAYOUT,
              formats=FORMATS, checkpoint=config['checkpoint'], data=config['data'],
              coverage=coverage_rows)
write_json(QC_DIR / 'notebook_run_qc.json', run_qc)
display(pd.DataFrame(coverage_rows))
print('PASS: source hashes unchanged, all selected-feature points retained, no axis clipping.')
print('Saved QC data only:', QC_DIR)